[How to Build a Local AI Agent With Python (Ollama, LangChain & RAG)](https://youtu.be/E4l91XKQSgw?si=PtR5Z37ImgNYVdIP)

Testare ollama

In [7]:
from ollama import chat

In [8]:

response = chat(
    model='gemma3',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
# print(response.message.content)

In [9]:
print(response.message.content)

Hello there! How's your day going so far? 😊 

Is there anything I can help you with today? Do you want to:

*   Chat about something?
*   Get some information?
*   Play a game?
*   Something else entirely?


Ollama image input test

In [ ]:
import base64

from ollama import chat
from pathlib import Path

while True:
    path  = input("Plase enter the path to the image:")
    if path =="q":
        break

    response = chat(
    model='gemma3',
    messages =[
            {
            'role':'user',
            # 'content':'What is in this image? Be concise.',
            'content':input("Please enter the question about image you send to llm:"),
            'images':[path]
            }
        ],
    )
    print(response.message.content)

# img=  base64.b64encode(Path(path).read_bytes().decode())



# /home/petrisor/Pictures/quilia-lbqZUefMLvQ-unsplash.jpg
# /home/petrisor/Pictures/Webcam/2026-08-11-133819.jpg
# /home/petrisor/Pictures/Webcam/2026-08-11-134013.jpg
# /home/petrisor/Pictures/Webcam/2026-08-11-134405.jpg
# /home/petrisor/Pictures/Webcam/2026-08-11-134351.jpg

# /home/petrisor/Pictures/Webcam/2026-08-11-141114.jpg
# /home/petrisor/Pictures/Webcam/2026-08-11-141133.jpg
# /home/petrisor/Pictures/Webcam/
# /home/petrisor/Pictures/Webcam/
# /home/petrisor/Pictures/channels4_profile.jpg



`ollama pull mxbai-embed-large`

In [8]:
import ollama

response = ollama.embed(
    model='mxbai-embed-large',
    input='The sky is blue because of Rayleigh scattering',
)
print(response.embeddings)

[[-0.03649428, 0.013126466, 0.011481987, -0.03743571, -0.042418975, 0.021661261, 0.0030563031, 0.033832118, 0.051979546, 0.058438092, -0.00218728, -0.011489711, 0.026925482, 0.011900986, -0.02310217, -0.023143476, -0.015152412, 0.005745816, -0.030849991, 0.041153073, -0.032377347, 0.020267643, -0.06465186, -0.05606616, 0.028726289, 0.051226035, 0.02071711, 0.012544443, 0.021674579, 0.03878167, -0.0232481, -0.021448692, -0.024594475, -0.050099254, 0.0053553986, 0.028541299, 0.00923221, -0.058304273, 0.01882048, -0.032470446, 0.04378232, -0.051518377, 0.02747076, -0.047464456, 0.0020603025, -0.046586003, -0.016128847, -0.024969049, 0.012312432, 0.02400297, 0.019118201, 0.039183773, 0.03761573, -0.040805645, -0.018626848, 0.048090566, -0.02349562, 0.0087943, 0.007173939, 0.002344994, -0.037633546, 0.0066240593, 0.013343238, -0.034506656, 0.0049440786, 0.054544516, -0.00687425, 0.0043579945, -0.009267743, -0.008383437, 0.0011990994, 0.025461126, 0.018982766, -0.0021638693, -0.022787023, 0.

Main

In [11]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate


model = OllamaLLM(model="gemma3")

template = """
You are an expert in answering questions about a pizza restaurant

Here are some relevant reviews: {reviews}

Here is the question to answer: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = prompt | model  

result=chain.invoke({"reviews": [],"question":"What is the best pizza place in town?"})

print(result)

Okay, based on the information I have (which, currently, is just that you've provided me with a placeholder for reviews!), I can’t definitively tell you the *best* pizza place in town. 

However, to give you the most helpful answer, I need that review data! 

**Once you provide me with the reviews, I’ll be able to analyze them and tell you which pizza place is consistently praised, based on factors like:**

*   **Taste:** What kind of toppings and crust are people raving about?
*   **Service:** Are the staff friendly and efficient?
*   **Value:** Are people getting a good deal for the price?
*   **Overall Experience:** What’s the general sentiment about the restaurant?

**So, please paste the reviews here, and I’ll do my best to identify the top pizza spot in town!** 

Do you want me to also consider other factors like delivery speed or atmosphere when I analyze the reviews?


In [ ]:
from vector import retriever

while True:
    print("\n\n-----------------------------------------")
    question = input("Ask your question(q to quit): ")
    print("\n\n")
    if question == "q":
        break

    reviews = retriever.invoke(question)
    result=chain.invoke({"reviews": reviews,"question":question})
    print(result)

ValueError: Could not connect to tenant default_tenant. Are you sure it exists?

Vector search

In [25]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import os
import pandas as pd

df = pd.read_csv("realistic_restaurant_reviews.csv")

embddings = OllamaEmbeddings(model = "mxbai-embed-large")

db_location = "./chrome_langchain_db"

add_documents = not os.path.exists(db_location)

if add_documents:
    documents = []
    ids = []

    for i , row in df.iterrows():
        document =  Document(
            page_content=row["Title"]+" "+row["Review"],
            metadata = {"rating":row["Rating"],"data":row["Date"]},
            id = str(i)
        )
        ids.append(str(i))
        documents.append(document)

vector_store = Chroma(
    collection_name="resturant_reviews",
    persist_directory=db_location,
    embedding_function=embddings
)

if add_documents:
    vector_store.add_documents(documents=documents,ids=ids)

retriever =vector_store.as_retriever(
    search_kwargs={"k":5}
)


ValueError: Could not connect to tenant default_tenant. Are you sure it exists?